# 🎙️ KionTTS Expressive Inference & Quality Testing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/C1ph3r404/KionTTS/blob/main/Training_Architecture/KionTTS_Inference_Colab.ipynb)

Test and evaluate your trained **KionTTS (Stage 2 Style Diffusion)** model directly in Google Colab or Kaggle!
This notebook:
1. Automatically downloads the **best Stage 2 checkpoint** (`kion_stage2_best.pth`) from your **Hugging Face Model Hub**.
2. Runs emotional and tag-conditioned speech synthesis across multiple intensity levels and style blends.
3. Renders interactive audio players inline so you can listen to generated speech samples directly.
4. Provides an interactive playground for synthesizing any custom text with emotional tags.


## 1. Hardware & GPU Check


In [ ]:
import torch
import subprocess

print("=" * 60)
print("Hardware & Accelerator Environment Check")
print("=" * 60)

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[✓] GPU Available: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    device = torch.device("cpu")
    print("[!] GPU not detected. Running on CPU (synthesis will be slower).")
    print("    To enable GPU in Colab: Runtime > Change runtime type > T4 GPU")


## 2. Dependencies & Codebase Setup


In [ ]:
import os
import sys
import shutil
import subprocess

def find_or_setup_repo():
    # Check if model/ exists in known or relative locations
    candidates = [
        os.path.abspath("."),
        os.path.abspath(".."),
        "/content/KionTTS",
        "/kaggle/working/KionTTS",
        "/kaggle/working",
    ]
    for cand in candidates:
        if os.path.exists(os.path.join(cand, "model")):
            return cand

    # Determine safe clone target
    if os.path.exists("/content"):
        target_dir = "/content/KionTTS"
    elif os.path.exists("/kaggle/working"):
        target_dir = "/kaggle/working/KionTTS"
    else:
        target_dir = os.path.abspath("./KionTTS")

    if not os.path.exists(os.path.join(target_dir, "model")):
        print(f"[*] Cloning KionTTS codebase to {target_dir}...")
        os.makedirs(os.path.dirname(target_dir), exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/C1ph3r404/KionTTS.git", target_dir], check=True)
    return target_dir

REPO_DIR = find_or_setup_repo()
print(f"[✓] KionTTS repository root: {REPO_DIR}")

STYLETTS2_DIR = os.path.join(REPO_DIR, "StyleTTS2")

for p in [REPO_DIR, STYLETTS2_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 2. System and Python dependencies
print("[*] Installing audio & synthesis dependencies (phonemizer, soundfile, munch, huggingface_hub, monotonic_align)...")
subprocess.run(["apt-get", "install", "-y", "espeak-ng"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "phonemizer", "soundfile", "munch", "huggingface_hub", "einops", "librosa", "pyyaml", "cython"
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/resemble-ai/monotonic_align.git"
], check=False)

print("[✓] Dependencies installed & environment ready!")


## 3. StyleTTS2 Pretrained Utility Assets
Downloads required pretrained text aligner, pitch extractor, and PL-BERT models if not already present.


In [ ]:
import urllib.request

def download_asset(url, dest, desc):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest) or os.path.getsize(dest) < 1024 * 1024:
        print(f"[*] Downloading {desc}...")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"    [✓] Saved: {dest}")
        except Exception:
            subprocess.run(["curl", "-L", "-o", dest, url], check=True)
            print(f"    [✓] Saved via curl: {dest}")
    else:
        print(f"[✓] {desc} verified ({os.path.getsize(dest)/(1024*1024):.1f} MB)")

# Download ASR, F0 (JDC), and PL-BERT
download_asset(
    "https://github.com/yl4579/StyleTTS2/raw/main/Utils/ASR/epoch_00080.pth",
    os.path.join(STYLETTS2_DIR, "Utils/ASR/epoch_00080.pth"),
    "ASR Text Aligner (epoch_00080.pth)"
)
download_asset(
    "https://github.com/yl4579/StyleTTS2/raw/main/Utils/JDC/bst.t7",
    os.path.join(STYLETTS2_DIR, "Utils/JDC/bst.t7"),
    "F0 Pitch Extractor (bst.t7)"
)
download_asset(
    "https://github.com/yl4579/StyleTTS2/raw/main/Utils/PLBERT/step_1000000.t7",
    os.path.join(STYLETTS2_DIR, "Utils/PLBERT/step_1000000.t7"),
    "PL-BERT Language Model (step_1000000.t7)"
)
print("[✓] All pretrained utility assets verified!")


## 4. Hugging Face Hub Authentication & Best Checkpoint Download
Fetches the best Stage 2 model (`kion_stage2_best.pth`) from your Hugging Face repository.


In [ ]:
import os
import getpass
from huggingface_hub import HfApi, hf_hub_download, login

# Repository configuration
HF_REPO_ID = "nate0001/KionTTS-Checkpoints"  # @param {type:"string"}

# HF Token Resolution
def get_hf_token():
    # 1. Colab Secrets
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    # 2. Environment variable
    t = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if t: return t.strip()
    return ""

hf_token = get_hf_token()
if not hf_token:
    print("[*] If your Hugging Face repository is private, enter your token below (press Enter to skip if public):")
    try:
        hf_token = getpass.getpass("HF Write/Read Token: ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        login(token=hf_token, add_to_git_credential=False)
        print("[✓] Authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"[!] HF Login notice: {e}")

if os.path.exists("/content"):
    LOCAL_CKPT_DIR = "/content/checkpoints"
elif os.path.exists("/kaggle/working"):
    LOCAL_CKPT_DIR = "/kaggle/working/checkpoints"
else:
    LOCAL_CKPT_DIR = os.path.join(REPO_DIR, "checkpoints")

os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

def download_hf_checkpoint(filename, repo_id=HF_REPO_ID, token=hf_token):
    local_path = os.path.join(LOCAL_CKPT_DIR, filename)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1024 * 1024:
        print(f"[✓] Checkpoint '{filename}' already exists locally ({os.path.getsize(local_path)/(1024*1024):.1f} MB).")
        return local_path
    print(f"[*] Fetching '{filename}' from Hugging Face [{repo_id}]...")
    try:
        dest = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            token=token or None,
            local_dir=LOCAL_CKPT_DIR,
            local_dir_use_symlinks=False,
        )
        print(f"[✓] Successfully downloaded from HF: {dest} ({os.path.getsize(dest)/(1024*1024):.1f} MB)")
        return dest
    except Exception as e:
        print(f"[-] '{filename}' could not be downloaded directly: {e}")
        return None

# Locate best checkpoint: try kion_stage2_best.pth first, then pointer, then latest
print(f"[*] Locating best Stage 2 checkpoint from {HF_REPO_ID}...")
best_ckpt_path = download_hf_checkpoint("kion_stage2_best.pth")

if not best_ckpt_path or not os.path.exists(best_ckpt_path):
    print("[*] Checking 'latest_stage2_checkpoint.txt' pointer...")
    ptr_file = download_hf_checkpoint("latest_stage2_checkpoint.txt")
    if ptr_file and os.path.exists(ptr_file):
        with open(ptr_file, "r") as f:
            target_name = f.readline().strip()
        if target_name:
            print(f"[*] Pointer references: {target_name}")
            best_ckpt_path = download_hf_checkpoint(target_name)

if not best_ckpt_path or not os.path.exists(best_ckpt_path):
    print("[*] Attempting fallback to 'kion_stage2_latest.pth'...")
    best_ckpt_path = download_hf_checkpoint("kion_stage2_latest.pth")

if not best_ckpt_path or not os.path.exists(best_ckpt_path):
    raise FileNotFoundError(
        f"No Stage 2 checkpoint found in HF repo '{HF_REPO_ID}'! "
        "Please verify the repo name and that your HF_TOKEN has access."
    )

print(f"
[★] Selected Checkpoint: {best_ckpt_path} ({os.path.getsize(best_ckpt_path)/(1024*1024):.1f} MB)")


## 5. Load KionTTS Expressive Architecture
Initializes the acoustic backbone, prosody predictor, vocoder decoder, and KionStyleAdapter with Stage 2 weights.


In [ ]:
import yaml
import torch
from munch import Munch

import importlib.util
infer_script_path = os.path.join(REPO_DIR, "Training_Architecture/colab_cells/08_inference_test.py")
spec = importlib.util.spec_from_file_location("inference_test", infer_script_path)
infer_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infer_mod)

CONFIG_PATH = os.path.join(STYLETTS2_DIR, "Configs/kion_config.yml")
if not os.path.exists(CONFIG_PATH):
    for c in [
        os.path.join(STYLETTS2_DIR, "Configs/config.yml"),
        os.path.join(REPO_DIR, "Configs/kion_config.yml")
    ]:
        if os.path.exists(c):
            CONFIG_PATH = c
            break

print(f"[*] Loading KionStyleTTS2 from: {best_ckpt_path}")
print(f"[*] Using Config: {CONFIG_PATH}")

model, config = infer_mod.load_kion_model(CONFIG_PATH, best_ckpt_path, device)
print("[✓] Model loaded and ready for synthesis!")


## 6. Interactive Emotion Testing & Evaluation Battery
Synthesizes a curated set of sentences across all core emotion tags and blends. Play audio directly in your browser!


In [ ]:
import os
import IPython.display as ipd
import soundfile as sf

EVAL_DIR = os.path.join(LOCAL_CKPT_DIR, "eval_samples")
os.makedirs(EVAL_DIR, exist_ok=True)
SAMPLE_RATE = 24000

test_cases = [
    {
        "title": "Neutral / Baseline Conversational",
        "tag": "[neutral]",
        "text": "Welcome to KionTTS. This voice is running directly in Google Colab with continuous style conditioning.",
        "fname": "01_neutral.wav",
        "pace": 1.0,
    },
    {
        "title": "High Energy Happy",
        "tag": "[happy=0.9]",
        "text": "I am absolutely thrilled to see that the training completed so wonderfully! Everything worked out!",
        "fname": "02_happy.wav",
        "pace": 1.05,
    },
    {
        "title": "Subtle Whisper",
        "tag": "[whisper=0.8,calm=0.6]",
        "text": "Keep your voice down, we do not want anyone to hear this secret.",
        "fname": "03_whisper.wav",
        "pace": 0.95,
    },
    {
        "title": "Intense / Frustrated Anger",
        "tag": "[angry=0.8]",
        "text": "I have told you multiple times that this cannot continue! We need to resolve this right now.",
        "fname": "04_angry.wav",
        "pace": 1.1,
    },
    {
        "title": "Melancholic / Sad",
        "tag": "[sad=0.7]",
        "text": "I really thought things would turn out differently this time. It is hard to see it end like this.",
        "fname": "05_sad.wav",
        "pace": 0.9,
    },
    {
        "title": "Playful & Teasing Blend",
        "tag": "[playful=0.7,teasing=0.6]",
        "text": "You seriously thought I would not notice that? Nice try!",
        "fname": "06_playful_teasing.wav",
        "pace": 1.0,
    },
    {
        "title": "Sarcastic & Deadpan",
        "tag": "[sarcasm=0.8,deadpan=0.5]",
        "text": "Oh, wonderful. Another meeting that definitely could have just been an email.",
        "fname": "07_sarcasm.wav",
        "pace": 1.0,
    },
    {
        "title": "Affectionate & Soothing",
        "tag": "[affectionate=0.8,soothing=0.7]",
        "text": "Do not worry, you did great today. Take a deep breath and rest easy.",
        "fname": "08_affectionate.wav",
        "pace": 0.95,
    }
]

print("=" * 70)
print(f"  Synthesizing {len(test_cases)} Emotion & Style Evaluation Samples...")
print("=" * 70)

for i, tc in enumerate(test_cases, 1):
    tag = tc["tag"]
    text = tc["text"]
    out_file = os.path.join(EVAL_DIR, tc["fname"])

    print(f"\n[{i}/{len(test_cases)}] {tc['title']}")
    print(f"     Tag  : {tag}")
    print(f"     Text : \"{text}\"")

    try:
        wav = infer_mod.synthesize(
            model=model,
            text=text,
            style_tag=tag,
            device=device,
            pace=tc.get("pace", 1.0),
            seed=42 + i,
        )
        sf.write(out_file, wav, SAMPLE_RATE)
        dur = len(wav) / SAMPLE_RATE
        print(f"     [✓] Generated ({dur:.2f}s) → {out_file}")

        # Display audio player with markdown label
        ipd.display(ipd.Markdown(f"**{tc['title']}** &mdash; `{tag}`"))
        ipd.display(ipd.Audio(wav, rate=SAMPLE_RATE))
    except Exception as e:
        print(f"     [!] Synthesis failed: {e}")

print("\n" + "=" * 70)
print(f"[✓] All samples generated and saved to: {EVAL_DIR}")
print("=" * 70)


## 7. Custom Interactive Speech Synthesis Playground
Type any sentence, select your emotional conditioning, and synthesize immediately!


In [ ]:
# @title 🎙️ Custom Synthesis Form
# @markdown Enter text and emotion parameters to synthesize on demand:

input_text = "This is KionTTS speaking with custom emotion conditioning. Let me know what you think!"  # @param {type:"string"}
emotion_tag = "[happy=0.8,playful=0.5]"  # @param ["[neutral]", "[happy=0.8]", "[whisper=0.8]", "[angry=0.8]", "[sad=0.7]", "[playful=0.7,teasing=0.5]", "[sarcasm=0.8,deadpan=0.5]", "[calm=0.8]", "[excited=0.9]", "[curious=0.8]", "custom"] {allow-input: true}
custom_tag = ""  # @param {type:"string"}
speaking_pace = 1.0  # @param {type:"slider", min:0.6, max:1.4, step:0.05}
random_seed = 42  # @param {type:"integer"}

effective_tag = custom_tag.strip() if emotion_tag == "custom" and custom_tag.strip() else emotion_tag

print(f"[*] Text  : \"{input_text}\"")
print(f"[*] Style : {effective_tag}")
print(f"[*] Pace  : {speaking_pace}x | Seed: {random_seed}")

wav = infer_mod.synthesize(
    model=model,
    text=input_text,
    style_tag=effective_tag,
    device=device,
    pace=speaking_pace,
    seed=random_seed,
)

custom_out = os.path.join(EVAL_DIR, "custom_synthesis.wav")
sf.write(custom_out, wav, SAMPLE_RATE)
print(f"[✓] Done! Audio duration: {len(wav)/SAMPLE_RATE:.2f}s")

ipd.display(ipd.Markdown(f"### 🔊 Synthesized Result: `{effective_tag}`"))
ipd.display(ipd.Audio(wav, rate=SAMPLE_RATE))
